# 02 — Score the Baseline (merged/generalist) Adapter

**Runs on:** Kaggle GPU T4. **Covers:** TASKS P6.1 (scoring only — NO training).

The merged/generalist baseline adapter `fyp-gemma3-1b-slm-merged-qlora` is ALREADY TRAINED
(via `FT_Merged_Adapter.ipynb`, r=16, α=16, same recipe as the three specialists). This notebook
only loads it and scores it through the NEW label-logit scoring path so it appears as config (a)
in the final results matrix — the specialist-vs-generalist control for the thesis hypothesis.

This adapter is the experimental control; it never runs alongside the specialists in the system.

In [ ]:
REPO_URL = "https://github.com/<YOUR_USER>/<YOUR_REPO>.git"  # TODO
!git clone -q {REPO_URL} slm_shield
%cd slm_shield
# Kaggle/Colab ship a pre-provisioned torch+CUDA. Do NOT `uv sync` here (it would rebuild the
# GPU stack and risk CUDA mismatch). Install the behaviour-critical libs on top of platform torch,
# pinned to the versions declared in pyproject.toml (the same ones your adapters were trained under).
!pip install -q "transformers==4.53.1" "unsloth==2025.7.2" peft trl accelerate bitsandbytes
# If Kaggle's preinstalled versions clash, restart the kernel after install and re-run from here.
from kaggle_secrets import UserSecretsClient
import os
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
ARTIFACT_REPO = "hirushafernando/slm-shield-artifacts"  # scores/manifests from notebook 01

In [ ]:
# Pull the split manifests produced by notebook 01
from huggingface_hub import snapshot_download
ART = snapshot_download(ARTIFACT_REPO, repo_type="dataset", token=HF_TOKEN)
from src.eval.splits import load_manifests
manifests = load_manifests(f"{ART}/manifests")

In [ ]:
# Load backbone + ONLY the baseline adapter (single adapter; it is a standalone classifier)
from src.model_loader import load_baseline_adapter
model, tokenizer = load_baseline_adapter(
    adapter_repo="hirushafernando/fyp-gemma3-1b-slm-merged-qlora", hf_token=HF_TOKEN)

In [ ]:
# Score the F-split through the new label-logit path (single probability, no OR/fusion).
# Uses the merged-instruction template variant frozen in src/templates.py.
from src.scoring import bulk_score_baseline
out = bulk_score_baseline(model, tokenizer, manifests, split="F",
                          out_path=f"{ART}/scores/baseline_F.parquet", resume=True)
print(out)  # AC: no training performed; r=16, alpha=16 confirmed identical to specialists (controlled comparison)

In [ ]:
# Persist back to the artefact repo
from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_folder(folder_path=f"{ART}/scores", repo_id=ARTIFACT_REPO,
                                    repo_type="dataset", path_in_repo="scores")
print("Baseline F-split scores persisted. Test-split baseline scoring happens in notebook 03.")